# Figure generation — Unified Senescence / SASP Gene Database

Regenerates all four manuscript figures from the pipeline outputs in `../results/`
and `../data/processed/`. Run top-to-bottom after `run_pipeline.py` (or `main.sh`)
has produced those files. Self-contained: no platform-specific dependencies.

Figures produced (written to `../figures/`):
1. `source_overlap.png` — 4-way source overlap (UpSet-style)
2. `expression_heatmap.png` — GTEx median expression, genes in >=3 sources
3. `interaction_network.png` — STRING high-confidence hub network
4. `coverage_gain.png` — per-layer coverage + redundancy removed


In [1]:
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

RESULTS = Path("../results"); PROC = Path("../data/processed"); RAW = Path("../data/raw")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})
META_GREY = "#8a8a8a"
SRC_PALETTE = {1: "#b8b8b8", 2: "#6baed6", 3: "#3182bd", 4: "#c44e52"}

db = json.loads((RESULTS / "senescence_sasp_database.json").read_text())
genes = db["genes"]
metrics = json.loads((RESULTS / "build_metrics.json").read_text())
SOURCES = db["metadata"]["sources"]
print(f"{len(genes)} genes loaded; sources = {SOURCES}")

1250 genes loaded; sources = ['CellAge', 'GenAge', 'SenMayo', 'Reactome']


## Figure 1 — Source overlap (UpSet-style)

In [2]:
member = {s: set() for s in SOURCES}
for gid, g in genes.items():
    for s in g["sources"]:
        member[s].add(gid)

combo_counts = defaultdict(int)
for gid, g in genes.items():
    combo_counts[tuple(sorted(g["sources"]))] += 1
top = sorted(combo_counts.items(), key=lambda kv: -kv[1])[:15]

fig, (axb, axm) = plt.subplots(2, 1, figsize=(11, 7),
    gridspec_kw={"height_ratios": [3, 2], "hspace": 0.05}, sharex=True)
xs = np.arange(len(top))
heights = [c for _, c in top]
ncols = [len(k) for k, _ in top]
axb.bar(xs, heights, color=[SRC_PALETTE[n] for n in ncols], edgecolor="white", linewidth=0.6)
for x, h in zip(xs, heights):
    axb.text(x, h + max(heights)*0.01, str(h), ha="center", va="bottom", fontsize=9)
axb.set_ylabel("Genes in intersection")
axb.set_title("Source overlap across four senescence/SASP seed lists")
axb.grid(axis="x", visible=False)

for yi, s in enumerate(SOURCES):
    axm.scatter(xs, [yi]*len(xs),
                c=[SRC_PALETTE[4] if s in k else "#dddddd" for k, _ in top], s=90, zorder=3)
for x, (k, _) in zip(xs, top):
    ys = [SOURCES.index(s) for s in k]
    if len(ys) > 1:
        axm.plot([x, x], [min(ys), max(ys)], color=SRC_PALETTE[4], lw=2, zorder=2)
axm.set_yticks(range(len(SOURCES))); axm.set_yticklabels(SOURCES)
axm.set_xticks(xs); axm.set_xticklabels([]); axm.set_ylim(-0.5, len(SOURCES)-0.5); axm.grid(False)
fig.text(0.5, 0.02, " | ".join(f"{s}: {len(member[s])}" for s in SOURCES),
         ha="center", color=META_GREY, fontsize=9)
fig.savefig(FIG / "source_overlap.png")
print("saved source_overlap.png; top:", [(k, c) for k, c in top[:6]])

saved source_overlap.png; top: [(('CellAge',), 711), (('GenAge',), 177), (('Reactome',), 99), (('SenMayo',), 90), (('CellAge', 'GenAge'), 81), (('CellAge', 'Reactome'), 28)]


## Figure 2 — GTEx expression heatmap (genes in ≥3 sources)

In [3]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# genes present in >=3 sources, with GTEx data
sel = [(gid, g) for gid, g in genes.items()
       if g["n_sources"] >= 3 and (g["expression"] or {}).get("gtex_median_tpm")]
tissues = sorted({t for _, g in sel for t in g["expression"]["gtex_median_tpm"]})
M = np.array([[g["expression"]["gtex_median_tpm"].get(t, 0.0) for t in tissues]
              for _, g in sel], dtype=float)
labels = [g["symbol"] for _, g in sel]
L = np.log2(M + 1.0)

# hierarchical clustering on both axes
row_order = leaves_list(linkage(pdist(L), method="average")) if len(L) > 2 else np.arange(len(L))
col_order = leaves_list(linkage(pdist(L.T), method="average")) if L.shape[1] > 2 else np.arange(L.shape[1])
Lo = L[np.ix_(row_order, col_order)]
rl = [labels[i] for i in row_order]
cl = [tissues[i].replace("_", " ") for i in col_order]

fig, ax = plt.subplots(figsize=(13, max(5, len(rl)*0.24)))
im = ax.imshow(Lo, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(cl))); ax.set_xticklabels(cl, rotation=90, fontsize=6)
ax.set_yticks(range(len(rl))); ax.set_yticklabels(rl, fontsize=6)
ax.set_title(f"GTEx v8 median expression — {len(rl)} genes in ≥3 sources")
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01); cb.set_label("log2(median TPM + 1)")
fig.savefig(FIG / "expression_heatmap.png")
print(f"saved expression_heatmap.png; matrix {Lo.shape}")

saved expression_heatmap.png; matrix (35, 54)


## Figure 3 — STRING high-confidence interaction network (top hubs)

In [4]:
import networkx as nx

edges = json.loads((PROC / "string_network.json").read_text())
G = nx.Graph()
for e in edges:
    G.add_edge(e["a"], e["b"], weight=e["score"])

sym2ns = {g["symbol"]: g["n_sources"] for g in genes.values()}
deg = dict(G.degree())
top_nodes = [n for n, _ in sorted(deg.items(), key=lambda kv: -kv[1])[:60]]
H = G.subgraph(top_nodes)

fig, ax = plt.subplots(figsize=(12, 10))
pos = nx.spring_layout(H, k=0.55, iterations=200, seed=42)
sizes = [deg[n]*6 + 40 for n in H.nodes()]
colors = [SRC_PALETTE.get(sym2ns.get(n, 1), "#b8b8b8") for n in H.nodes()]
nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.12, width=0.8)
nx.draw_networkx_nodes(H, pos, ax=ax, node_size=sizes, node_color=colors,
                       edgecolors="white", linewidths=0.5)
# label the 25 biggest hubs
lab = sorted(H.nodes(), key=lambda n: -deg[n])[:25]
nx.draw_networkx_labels(H, pos, {n: n for n in lab}, font_size=8, ax=ax)
ax.set_title("STRING v12 high-confidence network (score≥700) — top 60 hubs")
ax.axis("off")
# legend
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker='o', ls='', mfc=SRC_PALETTE[k], mec='white',
                  ms=10, label=f"{k} source"+("s" if k>1 else "")) for k in (1,2,3,4)]
ax.legend(handles=handles, title="Confirmed by", loc="lower left")
fig.savefig(FIG / "interaction_network.png")
top_hub = max(deg, key=deg.get)
print(f"saved interaction_network.png; top hub {top_hub} degree {deg[top_hub]}")

saved interaction_network.png; top hub TP53 degree 368


## Figure 4 — Coverage gain & redundancy removed

In [5]:
cov = metrics["coverage"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# panel A: annotation coverage per layer (net-new)
layers = [("Mouse\northolog","mouse_ortholog"), ("Zebrafish\northolog","zebrafish_ortholog"),
          ("GTEx\nexpression","gtex_expression"), ("HPA\nspecificity","hpa_specificity"),
          ("STRING\ninteraction","string_interaction"), ("All 3\nlayers","all_layers_complete")]
vals = [cov[k]["pct"] for _, k in layers]
ns = [cov[k]["n"] for _, k in layers]
bar_c = ["#3182bd"]*5 + ["#c44e52"]
bars = ax1.bar(range(len(layers)), vals, color=bar_c, edgecolor="white")
for i, (v, nn) in enumerate(zip(vals, ns)):
    ax1.text(i, v+0.8, f"{v}%\n({nn})", ha="center", va="bottom", fontsize=8)
ax1.set_xticks(range(len(layers))); ax1.set_xticklabels([l for l,_ in layers], fontsize=8)
ax1.set_ylabel("% of unified genes"); ax1.set_ylim(0, 108)
ax1.set_title("Net-new annotation coverage by layer"); ax1.grid(axis="x", visible=False)

# panel B: redundancy removed
summed = metrics["summed_entries"]; uniq = metrics["unique_genes"]; red = metrics["redundant_collapsed"]
ax2.bar(["Summed\nsource entries","Unified\nunique genes"], [summed, uniq],
        color=["#b8b8b8","#c44e52"], edgecolor="white")
ax2.text(0, summed+8, str(summed), ha="center", fontsize=10)
ax2.text(1, uniq+8, str(uniq), ha="center", fontsize=10)
ax2.annotate(f"-{red}\n({metrics['redundancy_pct']}%)", xy=(1, uniq), xytext=(0.5, (summed+uniq)/2),
             ha="center", color="#c44e52", fontweight="bold", fontsize=11)
ax2.set_ylabel("Gene entries"); ax2.set_title("Redundancy removed by unification (RQ1)")
ax2.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIG / "coverage_gain.png")
print("saved coverage_gain.png")

saved coverage_gain.png


## Application Case 1 — Multi-evidence Gene Prioritization Score

Composite priority score per gene: **source confirmation** (n_sources × 25, max 100) + **network centrality** (normalised STRING degree × 50, max 50) + **evolutionary conservation** (mean mouse/zebrafish % identity × 0.25, max ~25). Top 20 genes shown as stacked horizontal bar chart; gene label colour encodes n_sources.

In [ ]:
import numpy as np
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42

# ── build priority score ───────────────────────────────────────────────────
all_degs = [g['interaction']['string_degree']
            for g in genes.values()
            if g.get('interaction') and g['interaction']]
max_deg = max(all_degs)

records = []
for gid, g in genes.items():
    deg  = g['interaction']['string_degree'] if g.get('interaction') and g['interaction'] else 0
    m_id = g['conservation']['mouse']['perc_id']     if g.get('conservation') and g['conservation'].get('mouse')     else None
    z_id = g['conservation']['zebrafish']['perc_id'] if g.get('conservation') and g['conservation'].get('zebrafish') else None
    cons = np.mean([v for v in [m_id, z_id] if v is not None]) if any(v is not None for v in [m_id, z_id]) else 0.0
    src_s = g['n_sources'] * 25
    net_s = (deg / max_deg) * 50
    con_s = cons * 0.25
    records.append((g['symbol'], g['n_sources'], src_s, net_s, con_s, src_s + net_s + con_s))

records.sort(key=lambda x: -x[5])
top20  = records[:20]
syms   = [r[0] for r in top20]
src_v  = np.array([r[2] for r in top20])
net_v  = np.array([r[3] for r in top20])
con_v  = np.array([r[4] for r in top20])
ns_v   = [r[1] for r in top20]

# ── plot ───────────────────────────────────────────────────────────────────
COMP_COLORS = {
    'Source\nconfirmation':     '#c44e52',
    'Network\ncentrality':      '#3182bd',
    'Evolutionary\nconservation': '#74c476',
}
fig, ax = plt.subplots(figsize=(10, 8))
ys = np.arange(len(top20))[::-1]          # top gene at top
lefts = np.zeros(len(top20))
for lbl, vals_arr, clr in zip(COMP_COLORS, [src_v, net_v, con_v], COMP_COLORS.values()):
    ax.barh(ys, vals_arr, left=lefts, color=clr, edgecolor='white', linewidth=0.6, label=lbl)
    lefts += vals_arr

for yi, r in zip(ys, top20):
    ax.text(r[5] + 1.5, yi, f'{r[5]:.0f}', va='center', fontsize=11)

ax.set_yticks(ys)
ax.set_yticklabels(syms, fontsize=11)
for tick, ns in zip(ax.get_yticklabels(), ns_v[::-1]):
    tick.set_color(SRC_PALETTE[ns])
    tick.set_fontweight('bold')

ax.set_xlabel('Priority score')
ax.set_xlim(0, float(lefts.max()) * 1.12)
ax.grid(axis='y', visible=False)

comp_legend = ax.legend(loc='lower right', fontsize=11,
                        title='Score component', title_fontsize=11)
from matplotlib.lines import Line2D
ns_handles = [Line2D([0],[0], marker='o', ls='', mfc=SRC_PALETTE[k], mec='white',
                     ms=10, label=f'{k} source'+('s' if k>1 else ''))
              for k in sorted(set(ns_v))]
ax.legend(handles=ns_handles, title='Confirmed by (label colour)',
          loc='center right', fontsize=11, title_fontsize=11)
ax.add_artist(comp_legend)

fig.tight_layout()
fig.savefig(FIG / 'priority_score.png')
print('saved priority_score.png; top gene:', top20[0][0], f'score={top20[0][5]:.1f}')

## Application Case 2 — Aging Tissue Expression Landscape of Top Hub Genes

GTEx v8 median expression (log2 TPM + 1) for the top 15 priority-scored senescence genes across 15 aging-relevant tissues. Rows and columns are independently hierarchically clustered. Left colour strip encodes source confirmation count.

In [ ]:
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

AGING_TISSUES = [
    'Brain_Cortex', 'Brain_Hippocampus', 'Heart_Left_Ventricle',
    'Kidney_Cortex', 'Liver', 'Lung', 'Muscle_Skeletal',
    'Skin_Sun_Exposed_Lower_leg', 'Adipose_Subcutaneous',
    'Adipose_Visceral_Omentum', 'Artery_Aorta', 'Artery_Coronary',
    'Colon_Transverse', 'Small_Intestine_Terminal_Ileum', 'Whole_Blood',
]
TISSUE_LABELS = [
    'Brain cortex', 'Brain hippocampus', 'Heart (LV)',
    'Kidney cortex', 'Liver', 'Lung', 'Skeletal muscle',
    'Skin (sun-exp.)', 'Adipose subcutaneous',
    'Adipose visceral', 'Aorta', 'Coronary artery',
    'Colon', 'Small intestine', 'Whole blood',
]

TOP15 = [r[0] for r in records[:15]]
sym2gene = {g['symbol']: g for g in genes.values()}

M = np.array([
    [sym2gene[s]['expression']['gtex_median_tpm'].get(t, 0.0) for t in AGING_TISSUES]
    for s in TOP15
], dtype=float)
L = np.log2(M + 1.0)

row_order = leaves_list(linkage(pdist(L),   method='average'))
col_order = leaves_list(linkage(pdist(L.T), method='average'))
Lo = L[np.ix_(row_order, col_order)]
rl = [TOP15[i]         for i in row_order]
cl = [TISSUE_LABELS[i] for i in col_order]
ns_ordered = [sym2gene[s]['n_sources'] for s in rl]

fig = plt.figure(figsize=(13, 6))
gs = fig.add_gridspec(1, 2, width_ratios=[0.03, 1], wspace=0.03)
ax_strip = fig.add_subplot(gs[0])
ax_heat  = fig.add_subplot(gs[1])

# colour strip (n_sources)
strip_arr = np.array([[ns] for ns in ns_ordered], dtype=float)
ax_strip.imshow(strip_arr, aspect='auto',
                cmap=mpl.colors.ListedColormap([SRC_PALETTE[k] for k in sorted(SRC_PALETTE)]),
                vmin=0.5, vmax=4.5)
ax_strip.set_xticks([])
ax_strip.set_yticks(range(len(rl)))
ax_strip.set_yticklabels(rl, fontsize=11)
ax_strip.tick_params(length=0)
ax_strip.spines[:].set_visible(False)

# heatmap
im = ax_heat.imshow(Lo, aspect='auto', cmap='viridis')
ax_heat.set_xticks(range(len(cl)))
ax_heat.set_xticklabels(cl, rotation=45, ha='right', fontsize=11)
ax_heat.set_yticks([])
ax_heat.grid(False)
cb = fig.colorbar(im, ax=ax_heat, fraction=0.025, pad=0.01)
cb.set_label('log2(median TPM + 1)', fontsize=11)

# legend
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker='s', ls='', mfc=SRC_PALETTE[k], mec='white',
                  ms=11, label=f'{k} source'+('s' if k>1 else ''))
           for k in sorted(SRC_PALETTE)]
ax_strip.legend(handles=handles, title='Confirmed by',
                loc='upper left', bbox_to_anchor=(0, -0.18),
                fontsize=11, title_fontsize=11, ncol=2)

fig.savefig(FIG / 'aging_tissue_landscape.png')
print(f'saved aging_tissue_landscape.png; matrix {Lo.shape}')